In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import html
import contractions
import nltk
import os

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import euclidean_distances

import joblib
import random


In [17]:
# text preprocessing objects
svd_model = joblib.load('../../pickle-object/svd.pkl')
tfidf_vectorizer = joblib.load('../../pickle-object/tfidf.pkl')
final_n_svs = joblib.load('../../pickle-object/final_n_svs.pkl')

# model prediction objects
X_columns = joblib.load('../../pickle-object/X_columns.pkl')
model = joblib.load('../../pickle-object/model.pkl')

# for clustering
ref_df = pd.read_csv('../../pickle-object/clusters.csv', index_col='Cluster_ID')

In [18]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_nltk_alpha_only(text):
    text = html.unescape(str(text))
    text = contractions.fix(text)

    tokens = word_tokenize(text.lower())

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word.isalpha() and word not in stop_words and len(word) > 2
    ]

    return " ".join(tokens)

In [19]:
def description_to_svd_vector(description):
    processed_text = preprocess_nltk_alpha_only(description)

    tfidf_vector = tfidf_vectorizer.transform([processed_text])

    svd_vector = svd_model.transform(tfidf_vector)[:, :final_n_svs]

    svd_vector_df = pd.DataFrame(
        svd_vector,
        columns=[f"SV_{i+1}" for i in range(final_n_svs)]
    )

    return svd_vector_df

In [20]:
svd_cols = [col for col in ref_df.columns if col.startswith('SV_')]
centroids_matrix = ref_df[svd_cols].values
competition_scores = ref_df['Competition_Score'].to_dict()

scaler = Normalizer(norm='l2')

def clustering(svd_vector_df):
    """Takes the SVD dataframe and instantly finds its market position."""
    
    new_scaled = scaler.transform(svd_vector_df.values)
    
    distances = euclidean_distances(new_scaled, centroids_matrix)[0]
    
    cluster = ref_df.index[np.argmin(distances)]
    competition_score = competition_scores[cluster]
    
    return cluster, competition_score

In [21]:
def predict_output(vec):
    return model.predict(pd.DataFrame(data=vec, columns=X_columns).fillna(0))

In [22]:
def run_pipeline(description=None, bin=False, bin_method=round):
    if description == None:
        description = input('Input game description: ')
        
    svd_df = description_to_svd_vector(description)

    predicted_score = predict_output(svd_df)
    if bin:
        predicted_score = bin_method(predicted_score[0])
    cluster, competition_score = clustering(svd_df)

    print(f"Predicted Score: {predicted_score}")
    print(f"Cluster: {cluster}, Competition Score: {competition_score}")

In [28]:
description = """
A two page d6 based freeform tabletop adventure game!
"""

run_pipeline(description, bin=True)

Predicted Score: 7
Cluster: 6, Competition Score: 1.0
